In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 21.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 345kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.21MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.8MB/s]


In [3]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print("Shape of X:", X.shape)   # [N, C, H, W]
    print("Shape of y:", y.shape)   # [N]
    print("y dtype:", y.dtype)
    break

Shape of X: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64])
y dtype: torch.int64


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X, y = next(iter(test_dataloader))
X, y = X.to(device), y.to(device)

logits = model(X)
print("logits shape:", logits.shape)   # 预计 [64, 10]

pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)

print("pred_probab shape:", pred_probab.shape)
print("y_pred shape:", y_pred.shape)
print("first 10 preds:", y_pred[:10])
print("first 10 labels:", y[:10])

logits shape: torch.Size([64, 10])
pred_probab shape: torch.Size([64, 10])
y_pred shape: torch.Size([64])
first 10 preds: tensor([2, 8, 2, 7, 7, 6, 7, 6, 0, 7])
first 10 labels: tensor([9, 2, 1, 1, 6, 1, 4, 6, 5, 7])


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()      # 先清梯度
        pred = model(X)            # forward
        loss = loss_fn(pred, y)    # 计算 loss
        loss.backward()            # backward
        optimizer.step()           # 更新参数

        if batch % 100 == 0:
            loss_value = loss.item()
            current = (batch + 1) * len(X)
            print(f"loss: {loss_value:>7f}  [{current:>5d}/{size:>5d}]")

In [8]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()

    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)

            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(f"Test Error:\n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

In [10]:
epochs = 5   # 先用 1 跑通，再改成 5

for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)

print("Done!")

Epoch 1
-------------------------------
loss: 2.169019  [   64/60000]
loss: 2.152690  [ 6464/60000]
loss: 2.103620  [12864/60000]
loss: 2.114050  [19264/60000]
loss: 2.063272  [25664/60000]
loss: 2.015301  [32064/60000]
loss: 2.027246  [38464/60000]
loss: 1.958475  [44864/60000]
loss: 1.957138  [51264/60000]
loss: 1.885002  [57664/60000]
Test Error:
 Accuracy: 60.4%, Avg loss: 1.886447

Epoch 2
-------------------------------
loss: 1.918692  [   64/60000]
loss: 1.881923  [ 6464/60000]
loss: 1.773913  [12864/60000]
loss: 1.804339  [19264/60000]
loss: 1.701878  [25664/60000]
loss: 1.659556  [32064/60000]
loss: 1.665256  [38464/60000]
loss: 1.581539  [44864/60000]
loss: 1.595525  [51264/60000]
loss: 1.487591  [57664/60000]
Test Error:
 Accuracy: 61.2%, Avg loss: 1.515027

Epoch 3
-------------------------------
loss: 1.580880  [   64/60000]
loss: 1.543464  [ 6464/60000]
loss: 1.399912  [12864/60000]
loss: 1.464129  [19264/60000]
loss: 1.355823  [25664/60000]
loss: 1.349140  [32064/60000]
